### Linda Zier
### ST 554
### Project 2

This project has two parts.  In the first part I have created a data quality class in pyspark. This class is a python class that wraps an SQL DataFrame and provides functionality for cleaning and checking data.

In the second part of the project I analyze data using spark SQL style dataframes and pandas-on-spark data frames.



In [17]:

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [18]:
import sys
sys.path.insert(0,'.')

import importlib
import Spark_class
importlib.reload(Spark_class)

from Spark_class import SparkDataCheck

In [19]:

# Loading the air data using the from_csv method
air_df = Spark_class.SparkDataCheck.from_csv("data/air.csv", spark)
           
air_df.df.show(5)



+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-22 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-22 21:00:00|   2.2|       137

26/03/22 10:55:25 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


### Examples:

- Here is an example of using my methods on the air quality data to check between 2 values.

In [20]:
air_df.check_col_range(column="T", lower=10.0, upper=11.1)
air_df.df.show(10)

26/03/22 10:55:25 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-22 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-22 21:00:00|   2.2|       137

- Now I'm showing what happens when you don't have at least a lower or an upper.

In [21]:
air_df.check_col_range(column="T")
#air_df.df.show()

DataFrame[_c0: int, Date: string, Time: timestamp, CO(GT): double, PT08.S1(CO): int, NMHC(GT): int, C6H6(GT): double, PT08.S2(NMHC): int, NOx(GT): int, PT08.S3(NOx): int, NO2(GT): int, PT08.S4(NO2): int, PT08.S5(O3): int, T: double, RH: double, AH: double, range_flag: boolean]

- Here I've supplied only an upper range.

In [22]:
air_df.check_col_range(column="T", upper=11.2)
air_df.df.show(10)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-22 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-22 21:00:00|   2.2|       137

26/03/22 10:55:26 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


- Here I've tested something that wasn't one of the number types we specified.

In [23]:
air_df.check_col_range(column="Date", lower=0)


Column 'Date' is not numeric.


In [24]:
air_df.df.dtypes

[('_c0', 'int'),
 ('Date', 'string'),
 ('Time', 'timestamp'),
 ('CO(GT)', 'double'),
 ('PT08.S1(CO)', 'int'),
 ('NMHC(GT)', 'int'),
 ('C6H6(GT)', 'double'),
 ('PT08.S2(NMHC)', 'int'),
 ('NOx(GT)', 'int'),
 ('PT08.S3(NOx)', 'int'),
 ('NO2(GT)', 'int'),
 ('PT08.S4(NO2)', 'int'),
 ('PT08.S5(O3)', 'int'),
 ('T', 'double'),
 ('RH', 'double'),
 ('AH', 'double')]

In [25]:
air_df.df.toPandas().isnull().sum()

26/03/22 10:55:26 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


_c0              0
Date             0
Time             0
CO(GT)           0
PT08.S1(CO)      0
NMHC(GT)         0
C6H6(GT)         0
PT08.S2(NMHC)    0
NOx(GT)          0
PT08.S3(NOx)     0
NO2(GT)          0
PT08.S4(NO2)     0
PT08.S5(O3)      0
T                0
RH               0
AH               0
dtype: int64

- Below I am checking for NULL values in the T (temperature) column

In [26]:
air_df.check_null(column="T")

DataFrame[_c0: int, Date: string, Time: timestamp, CO(GT): double, PT08.S1(CO): int, NMHC(GT): int, C6H6(GT): double, PT08.S2(NMHC): int, NOx(GT): int, PT08.S3(NOx): int, NO2(GT): int, PT08.S4(NO2): int, PT08.S5(O3): int, T: double, RH: double, AH: double, null_flag: boolean]

In [27]:
air_df.col_minmax()

26/03/22 10:55:26 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: 
 Schema: _c0
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


,col_name,min,max
0,_c0,0.0,9356.000
1,CO(GT),-200.0,11.900
2,PT08.S1(CO),-200.0,2040.000
3,NMHC(GT),-200.0,1189.000
4,C6H6(GT),-200.0,63.700
5,PT08.S2(NMHC),-200.0,2214.000
6,NOx(GT),-200.0,1479.000
7,PT08.S3(NOx),-200.0,2683.000
8,NO2(GT),-200.0,340.000
9,PT08.S4(NO2),-200.0,2775.000
